# 05 — Index C (Energy-Pulse)

**Objetivo.** Predecir Index_C explotando su relación con macro_factors.

**Pistas del enunciado.** Energy-Pulse — ligado a energía global y macro (oro, crudo, tipos de interés).

**Nivel de esfuerzo.** MEDIO. Las features auxiliares son la palanca; no explorar arquitecturas complejas.

**Estrategia.** LSTM + features de macro alineadas con `align_aux_features`. Las features útiles se identifican en `00_carga_y_EDA`. Si la correlación es débil, el baseline puede ganar de todas formas.

**Qué esperamos.** Mejoría sobre baseline al añadir macro. Backtest decide — no el loss.

**Qué NO hace.** No toca network_metrics ni news.

**Inputs.** `data/train_indices.csv`, `data/train_macro_factors.csv`, `data/test_macro_factors.csv`, `results/baselines.json`

**Output esperado.** `models/{OWNER}_Index_C.keras`, `results/index_C.json`

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

OWNER = "oscar"   # <-- cada miembro pone su nombre aquí
IDX   = "Index_C"

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from utils import (
    load_hackathon_data, make_window_dataset, make_temporal_split,
    align_aux_features, build_model, train_model,
    backtest_autoregressive, plot_history, plot_rollout,
    DATA_DIR, VAL_DAYS, V_IN_SHARED, RANDOM_SEED
)

os.makedirs('models',  exist_ok=True)
os.makedirs('results', exist_ok=True)
MODEL_PATH = f'models/{OWNER}_{IDX}.keras'

## 1. Carga + referencia baselines

In [ ]:
data  = load_hackathon_data(DATA_DIR)
idx   = data['train_indices']
serie = idx[IDX].dropna().values

with open('results/baselines.json') as f:
    baselines = json.load(f)
print(f'Baselines para {IDX}:')
for bl, rmse in baselines[IDX].items():
    print(f'  {bl:<15} RMSE = {rmse:,.0f}')

## 2. Alineamiento y selección de features macro

`align_aux_features(train_indices_df, aux_df, feature_cols)` alinea por fecha con ffill+bfill.

**Traer de `00_carga_y_EDA`:** las columnas de macro con mayor correlación con C.

In [ ]:
if 'train_macro' not in data:
    raise RuntimeError('train_macro_factors.csv no encontrado — necesario para C')

# TODO sábado: filtrar a las columnas que correlacionen según EDA (notebook 00). Placeholder = todas.
# Ver CLAUDE.md → § Deuda técnica, punto 1. Al editar esta línea, el JSON y el 09 se ajustan solos.
MACRO_FEATURES = list(data['train_macro'].columns)   # <-- filtrar al resultado del EDA

aux_train = align_aux_features(idx, data['train_macro'], MACRO_FEATURES)
print('Features macro alineadas:', list(aux_train.columns))
print('Shape aux_train:', aux_train.shape, '  Shape idx:', idx.shape)

## 3. Dataset con features auxiliares

In [ ]:
# Extraer valores de aux alineados con la serie de C
aux_values = aux_train.loc[idx.index].values   # (T, k)

serie_train, _ = make_temporal_split(serie, val_days=VAL_DAYS, v_in=V_IN_SHARED)
aux_train_only = aux_values[:len(serie_train)]

X, y = make_window_dataset(serie_train, V_IN_SHARED, use_log_rets=True,
                            aux_features=aux_train_only)

cut = int(len(X) * 0.8)
X_tr, y_tr = X[:cut], y[:cut]
X_v,  y_v  = X[cut:], y[cut:]

n_features = X.shape[2]
print(f'X_tr={X_tr.shape}  n_features={n_features}  (1 precio + {n_features-1} macro)')

## 4. LSTM con features macro

In [ ]:
import tensorflow as tf
tf.random.set_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

model = build_model('lstm', V_IN_SHARED, n_features=n_features)
hist  = train_model(model, X_tr, y_tr, X_v, y_v, epochs=300)
plot_history(hist, title=f'{IDX} — LSTM + macro')

## 5. Backtest con features de test_macro

`backtest_autoregressive` con `aux_data` usa las features del período de validación (simula usar `test_macro` en producción). El `aux_data` debe tener la misma longitud que la serie completa.

In [ ]:
predict_fn = lambda x: model.predict(x, verbose=0).ravel()[0]

# aux_data debe alinearse con la serie completa (misma longitud que serie)
# Para el backtest interno usa los últimos VAL_DAYS de aux_values como período val
bt = backtest_autoregressive(
    predict_fn, serie,
    log_ret_mode=True,
    aux_data=aux_values   # longitud == len(serie) — assert dentro de backtest lo verifica
)
print(f'LSTM+macro  RMSE backtest = {bt["rmse"]:,.0f}  |  dir_acc = {bt["dir_accuracy"]:.2%}')

# Comparar con mejor baseline
best_bl_rmse = min(baselines[IDX].values())
mejora = (best_bl_rmse - bt['rmse']) / best_bl_rmse * 100
print(f'Mejora sobre mejor baseline: {mejora:+.1f}%')

plot_rollout(serie, bt['preds'], index_name=f'{IDX} — LSTM+macro', val_days=VAL_DAYS)

## 6. Decisión final y guardado

In [ ]:
# Si LSTM+macro no mejora el baseline, cambiar approach_type a 'baseline_flat' etc.
# y no guardar model_path

best_bl_name = min(baselines[IDX], key=baselines[IDX].get)
_tipo_map = {'flat': 'baseline_flat', 'drift': 'baseline_drift', 'random_walk': 'baseline_rw'}

if bt['rmse'] < best_bl_rmse:
    model.save(MODEL_PATH)
    approach = 'nn'
    final_rmse = bt['rmse']
    final_path = MODEL_PATH
    notes = f'LSTM + {n_features-1} features macro'
else:
    approach   = _tipo_map[best_bl_name]
    final_rmse = best_bl_rmse
    final_path = None
    notes      = f'Baseline {best_bl_name} — macro no aportó mejora en backtest'

info = {
    'index':              IDX,
    'owner':              OWNER,
    'approach_type':      approach,
    'strategy':           approach,
    'rmse_backtest_252d': final_rmse,
    'model_path':         final_path,
    'log_ret_mode':       True if approach == 'nn' else False,
    'v_in':               V_IN_SHARED if approach == 'nn' else None,
    'n_features':         int(n_features) if approach == 'nn' else 1,
    'aux_source':         'train_macro'  if approach == 'nn' else None,
    'aux_test_source':    'test_macro'   if approach == 'nn' else None,
    'aux_columns':        MACRO_FEATURES if approach == 'nn' else None,
    'ghost_source_index': None,
    'ghost_lag':          None,
    'notes':              notes
}

with open('results/index_C.json', 'w') as f:
    json.dump(info, f, indent=2)

print('Guardado: results/index_C.json')
print(json.dumps(info, indent=2))